# EEG seizure detection on CHB-MIT — patient-specific diagnostic

Thin notebook: all logic lives in the `eia` package and `scripts/xylo_verify.py`. Runs **locally** or on **Google Colab** (faster network — CHB-MIT's ~40MB/record EDF files were the bottleneck for the local run this was built from, see `docs/eeg_seizure_results.md`).

**Why this notebook exists.** The subject-independent (deployment) result on CHB-MIT was ~chance (float balanced accuracy 0.525 +/- 0.030, `docs/eeg_seizure_results.md`). That alone can't tell "too little cross-patient data" apart from "the montage/band-pass/timestep front-end destroyed the seizure signal" — both would look identical from the outside. This notebook runs a **patient-specific diagnostic** (a near-free upper-bound test: split by *record* within one patient, instead of by *patient*) side by side with the subject-independent split, on the same pulled data, and prints an explicit verdict:

- Patient-specific **learns** (AUROC well above 0.5) but subject-independent doesn't -> **scale problem** — more patients/records is the confirmed fix.
- Patient-specific is **also** ~chance on the same records -> **front-end too lossy** — revisit montage/band/timesteps before downloading anything else.

Patient-specific is a **diagnostic, not the deployment metric** — CHB-MIT's own classic "high AUROC" results are usually patient-specific and do not reflect a device that has never seen the patient. Don't headline it.

## Setup
On Colab this clones the repo + installs deps (including the `data` extra for `wfdb`, needed to stream CHB-MIT). Locally it's a no-op if `eia` already imports.

In [ ]:
import importlib.util, sys, subprocess, os

ON_COLAB = 'google.colab' in sys.modules

if ON_COLAB:
    REPO = 'https://github.com/dcaglar-28/NeuroSoC.git'
    # Clone INTO a folder named `eia` regardless of the repo's own name
    # (this repo is `NeuroSoC` on GitHub) -- every path below assumes `eia/`.
    if not os.path.exists('eia'):
        subprocess.run(['git', 'clone', REPO, 'eia'], check=True)
        subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', '-e', 'eia[data]'], check=True)
        # xylo (rockpool/samna) is best-effort: samna often has no wheel for
        # Colab's Python version, and this notebook's float-model diagnostic
        # doesn't need XyloSim -- don't let a failed xylo install abort setup.
        subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', '-e', 'eia[xylo]'])
    src_paths = ('eia/src', 'eia/scripts')
else:
    src_paths = (os.path.abspath('../src'), os.path.abspath('../scripts'))

# ALWAYS ensure our repo's src/scripts are on sys.path, regardless of
# whether some "eia" was already importable (e.g. a stale editable install
# from an earlier cell run in this kernel, or an unrelated same-named
# package) -- a previous version of this cell skipped this on that path,
# which is exactly how the wrong "eia" (missing __version__) got imported.
for p in src_paths:
    if p not in sys.path:
        sys.path.insert(0, p)

# sys.path changes above don't retroactively fix an "eia" module already
# cached in sys.modules from an earlier, differently-configured run in this
# same kernel session (Colab keeps kernel state across cell re-runs). Evict
# a wrong one -- recognizable by missing __version__, which our real
# eia/__init__.py always defines -- before importing for real.
if 'eia' in sys.modules and not hasattr(sys.modules['eia'], '__version__'):
    del sys.modules['eia']

import eia
print('eia', eia.__version__, '--', eia.__file__)

## Parameters

`SUBJECTS` excludes the 8 CHB-MIT subjects confirmed (via a header-only probe, `docs/eeg_seizure_task.md` Part 0) to raise a reproducible `wfdb.io.convert.edf.read_edf` "math domain error": `chb12, chb13, chb14, chb16, chb17, chb18, chb19, chb21` (chb21 is also the same patient as chb01, so its loss doesn't cost a distinct patient).

Defaults here are set **higher than the local run** that produced `docs/eeg_seizure_results.md` (which used 9 subjects x 1 seizure record each due to slow local network) — Colab's connection should comfortably support pulling every available seizure record for all 9 loadable subjects, plus some non-seizure background. Increase `SEIZURE_RECORDS_PER_SUBJECT` further (CHB-MIT has 3-7 seizure records for most of these subjects) if runtime allows; this is exactly the "more patients/records" lever the verdict below might tell you to pull.

In [ ]:
SUBJECTS = ['chb01', 'chb02', 'chb03', 'chb05']   # FAST START (skips chb06, which stalls).
# Full decisive run: add 'chb06','chb07','chb08','chb09','chb10' back.
# The Drive cache makes scaling incremental -- adding subjects only downloads the new ones.
SEIZURE_RECORDS_PER_SUBJECT = 2     # >=2 required for the patient-specific split (holds out whole records)
NONSEIZURE_RECORDS_PER_SUBJECT = 1
N_SEEDS = 5                          # >=5 per the repo's multi-seed discipline (single-seed AUROC is noise)
EPOCHS = 20
N_RESTARTS = 5
MAX_VERIFY = 300

REQUIRE_REAL = True   # fail loudly instead of silently substituting synthetic EEG


## Persistent cache (optional, Colab only)

CHB-MIT downloads take a while even on Colab's connection, and a Colab runtime can reset (idle timeout, disconnect) mid-pull. This cell mounts Google Drive and points `CACHE_DIR` at a folder there, so `load_chbmit`'s per-record cache (`<subject>_<record>_montage.npz`) survives a runtime reset — a rerun after a reset resumes from whatever was already cached instead of re-downloading everything (see `load_chbmit`'s per-record progress output below: cached records print as instant "cache hit"s).

Guarded (`try/except` on `import google.colab`) so this is a no-op — `CACHE_DIR` just stays the local default — everywhere else, including plain local runs.

In [ ]:
CACHE_DIR = 'data/chbmit'  # local default

try:
    import google.colab  # noqa: F401
    from google.colab import drive
    drive.mount('/content/drive')
    CACHE_DIR = '/content/drive/MyDrive/eia_chbmit'
    print('Colab detected -- caching CHB-MIT downloads to', CACHE_DIR)
except ImportError:
    print('Not on Colab -- caching to local', CACHE_DIR)

import os
os.makedirs(CACHE_DIR, exist_ok=True)

## Load CHB-MIT once

Both splits below reuse this SAME loaded `data` object (no re-downloading per split, per patient, or per seed). Downloads are cached per-record to `CACHE_DIR` (`.edf` files themselves are also gitignored if `CACHE_DIR` is inside the repo — `wfdb`'s EDF reader additionally caches full local copies into the working directory as a side effect, harmless but not something to commit).

`load_chbmit` is resumable and watchable: it prints the full download plan up front (total/cached/to-download), then per-record progress with elapsed time and a running window count, and wraps every network call in a wall-clock timeout (`record_timeout_sec`, default 300s) with one retry — a stalled PhysioNet connection gets skipped (fewer windows from that record), never hangs the cell forever.

In [ ]:
from eia.datasets import load_chbmit

data = load_chbmit(
    subjects=SUBJECTS,
    seizure_records_per_subject=SEIZURE_RECORDS_PER_SUBJECT,
    nonseizure_records_per_subject=NONSEIZURE_RECORDS_PER_SUBJECT,
    cache_dir=CACHE_DIR,
)
print('X', data.X.shape, 'pos_frac', float(data.y.mean()))
print('patients:', sorted(set(data.groups.tolist())))

## Data card
Source, label definition, class balance, and the pediatric-epilepsy-not-TBI / imbalance / subject-independence caveats — printed once, for the whole pool both splits below draw from.

In [ ]:
from eia import report

card = report.data_card(data)
report.assert_provenance(card, data, 'eeg')

## Run BOTH splits on the same data

1. **Subject-independent** (the deployment metric) — split by patient (`case_level.split_data`, `data.groups`).
2. **Patient-specific** (the diagnostic) — split by record within each patient (`datasets.eeg_patient_specific_split`), skipping any patient with fewer than 2 records.

Both reuse the exact same training pipeline (`xylo_verify.train_modality`/`verify_modality`): same montage, band-pass, resample, delta encoding, `build_xylo_snn`, class-weighted CE, balanced-accuracy checkpoint selection. Only the split differs.

In [ ]:
from eia import case_level
from xylo_verify import (
    train_modality, verify_modality, _print_multiseed_summary, _stat_over,
    run_eeg_patient_specific, print_eeg_patient_specific_summary,
    print_eeg_diagnostic_verdict,
)

# --- 1. Subject-independent (deployment metric) ---
si_results = []
for seed in range(N_SEEDS):
    result = train_modality(
        'eeg', real=True, epochs=EPOCHS, n_hidden=63, threshold=0.25,
        batch_size=128, lr=1e-2, spike_reg=2e-2, seed=seed, n_restarts=N_RESTARTS,
        require_real=REQUIRE_REAL, loader=lambda **_kw: data,
        split_fn=case_level.split_data, card_verbose=False)
    result = verify_modality(result, max_verify=MAX_VERIFY)
    si_results.append(result)

_print_multiseed_summary('eeg', si_results)
si_bal_acc = _stat_over(si_results, 'float_bal_acc')

In [ ]:
# --- 2. Patient-specific (diagnostic) ---
ps_results = run_eeg_patient_specific(
    data, epochs=EPOCHS, n_hidden=63, threshold=0.25, batch_size=128, lr=1e-2,
    spike_reg=2e-2, n_restarts=N_RESTARTS, n_seeds=N_SEEDS, max_verify=MAX_VERIFY)
ps_summary = print_eeg_patient_specific_summary(ps_results)

## Verdict
Explicit side-by-side comparison and the decision rule from `docs/eeg_seizure_task.md`.

In [ ]:
print(f"Subject-independent float balanced acc: {si_bal_acc[0]:.3f} +/- {si_bal_acc[1]:.3f}")
if ps_summary:
    print(f"Patient-specific    float balanced acc: {ps_summary['bal_acc'][0]:.3f} +/- {ps_summary['bal_acc'][1]:.3f}")

print_eeg_diagnostic_verdict(ps_summary, si_bal_acc)

## Next

- If the verdict says **scale problem**: rerun with more subjects / more `SEIZURE_RECORDS_PER_SUBJECT` — this is the lever to pull, not a pipeline change.
- If the verdict says **front-end too lossy**: revisit `datasets.EEG_MONTAGE` (try 8 channels, the exact ceiling), the band-pass range, or `resample_to` (try more than 128 timesteps, trading off against the ECG quantization-fidelity lesson that fewer timesteps usually helps on-chip agreement) — *before* pulling more data, per `docs/eeg_seizure_task.md`.
- Update `docs/eeg_seizure_results.md` with whatever this run found (n_patients actually used will differ from the local run this notebook followed up on).
- Xylo deployment + bit-precise verification for a single trained net: `python scripts/xylo_verify.py --modality eeg --real` (see `docs/eeg_seizure_task.md`).